# npugraph_ex基础概念

欢迎来到 **npugraph_ex 基础功能** 部分！本章节系统讲解TorchAir的核心基础概念、整体架构，帮助读者建立完整的前置知识体系，为后续快速上手、功能调优、问题定位打下理论基础。
## 1.概述

TorchAir（Torch Ascend Intermediate Representation）是昇腾Ascend Extension for PyTorch（torch\_npu）的图模式能力扩展库，提供了昇腾设备亲和的torch.compile图模式后端，实现了PyTorch网络在昇腾NPU上的图模式**推理加速**以及**性能优化**。

TorchAir在Ascend Extension for PyTorch（torch\_npu）中的位置如[图1](#fig1)所示，图中左侧为单算子执行模式（Eager），右侧为torch.compile图执行模式（Graph）。torch.compile图模式不仅继承了大部分PyTorch原生的[Dynamo特性](https://docs.pytorch.org/docs/main/user_guide/torch_compiler/torch.compiler_dynamo_overview.html)（如动态shape图功能等），还在此基础上新增其他图优化和定位调试能力，例如FX图Pass优化、图内多流并行、集合通信算子入图等，课程中提到的概念请参考下方常用概念。

目前图执行分为两种模式：

- **基于npugraph\_ex后端的图模式（NPUGraph）**：通过设置torch.compile的backend="npugraph\_ex"开启，其采用Capture&Replay方式实现任务一次捕获多次执行。Capture阶段捕获Stream任务到Device侧，暂不执行；Replay阶段从Host侧发出执行指令，Device侧再执行已捕获的任务，从而减少Host调度开销，提升性能。

- **基于GE的图模式（Ascend IR）**：通过设置TorchAir的CompilerConfig实例属性 <b>mode="max-autotune"</b> 开启，其将PyTorch的FX计算图转换为昇腾中间表示（IR，Intermediate Representation），即Ascend IR计算图，并通过GE（Graph Engine，图引擎）实现计算图的编译和执行。

**本次课程核心聚焦、重点讲解`npugraph_ex`后端对应的NPUGraph图执行模式，本课程所有的功能、案例、调优手段均围绕该模式展开。**

**图 1**  TorchAir架构图

![](./images/torchair_architecture.png) <a id="fig1" ></a>


## 2.常用概念

本节列举了常用的术语和概念，以帮助开发者们更好地理解关键特性和实现原理。

| 名称               | 说明                                                                                                                                                                                                                                                                                                                                       |
|------------------|------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| Eager模式          | 单算子执行模式（未使用torch.compile），特点如下，单击[Link](https://pytorch.org/blog/optimizing-production-pytorch-performance-with-graph-transformations/)获取PyTorch官网介绍。即时执行：每个计算操作在定义后立即执行，无需构建计算图。动态计算图：每次运行可能生成不同的计算图。                                                                                                                                   |
| 图模式              | 一般指使用torch.compile加速的图执行方式，特点如下：延迟执行：所有计算操作先构成一张计算图，再在会话中下发执行。静态计算图：计算图在运行前固定。                                                                                                                                                                                                                                                           |
| torch_npu        | Ascend Extension for PyTorch，是PyTorch的昇腾插件，使昇腾NPU可以适配PyTorch框架，该插件的Python包名为torch_npu，文档中常以torch_npu作为Ascend Extension for PyTorch的简称。                                                                                                                                                                                                   |
| Pass             | 在深度学习框架（如PyTorch）和编译器（如TVM）中，Compiler Passes（编译器传递）和Partitioners（分区器）是优化图执行的关键技术，用于性能优化、硬件适配和计算图转换等，而Pass则是指在这些计算图上执行的特定变换操作。常见的Pass操作包括常量折叠、算子融合、内存优化等，单击[Link](https://docs.pytorch.org/executorch/stable/compiler-custom-compiler-passes.html)获取PyTorch官网详情。FX Pass是指对计算图（torch.fx.Graph）进行遍历、分析和转换等一系列操作，类似于传统编译器中的优化步骤（如常量折叠、算子融合）。 |
| ATen             | 全称为A Tensor Library，是PyTorch张量计算的底层核心函数库，这些函数通常称为ATen算子，负责所有张量操作（如加减乘除、矩阵运算、索引等）的C++实现，单击[Link](https://github.com/pytorch/pytorch/tree/main/aten/src/ATen)获取PyTorch官网介绍。                                                                                                                                                                |
| FX图              | Functionality Graph，PyTorch中用于表示模型计算流程的中间层数据结构。通过符号化追踪代码生成计算图，将Python代码转为中间表示（IR，Intermediate Representation），实现计算图的动态调整和优化（如量化、剪枝等），单击[Link](https://docs.pytorch.org/docs/stable/fx.html)获取torch.fx详情。                                                                                                                                 |
| In-place算子       | 原地算子，该类算子可直接修改输入数据，不创建新的存储空间。从而节省内存，避免复制数据的开销。                                                                                                                                                                                                                                                                                           |
| Out-of-place算子   | 非原地算子，又称“非In-place算子”，该类算子保持原始输入数据不变，会创建并返回新对象，带来额外存储开销。                                                                                                                                                                                                                                                                                 |
| 算子Schema         | 在PyTorch中，算子Schema（Operator Schema）定义了算子的输入、输出、属性以及行为规范，确保算子在正向传播（Forward）和反向传播（Backward）时能正确执行。PyTorch使用Schema来注册算子，并在编译或运行时进行验证。算子Schema主要通过修改native_functions.yaml文件实现，该文件位于PyTorch源码的aten/src/ATen目录下，用于声明算子的名称、参数类型、返回值类型及设备端实现函数。                                                                                                  |
| Ascend C         | CANN编程语言，原生支持C和C++标准规范，兼具开发效率和运行性能。基于Ascend C编写的算子程序，通过编译器编译和运行时调度，运行在昇腾AI处理器上。开发的算子简称为Ascend C算子，其调用方式一般为aclnnXxx的C接口形式，具体介绍请参考《[CANN Ascend C算子开发](https://hiascend.com/document/redirect/CannCommunityOpdevAscendC)》。                                                                                                                 |
| OpPlugin         | Ascend Extension for PyTorch（torch_npu）算子插件，为使用PyTorch框架的开发者提供便捷的NPU算子库调用能力，具体介绍参考Ascend/OpPlugin仓，而算子适配开发过程参考[Ascend Extension for PyTorch文档中心](https://hiascend.com/document/redirect/pytorchuserguide)中的《PyTorch 框架特性指南》“基于OpPlugin算子适配开发”章节。                                                                                         |
| TorchAir图模式    | PyTorch图模式（torch.compile）的一种实现，通过指定TorchAir为其backend的执行方式。                                                                                                                                                                                                                                                                               |
| GE               | Graph Engine，图引擎。它是计算图编译和运行的控制中心，提供图优化、图编译管理以及图执行控制等功能。GE通过统一的图开发接口提供多种AI框架的支持，不同AI框架的计算图可以实现到Ascend IR图的转换，单击[Link](https://www.hiascend.com/cann/graph-engine)获取详情。   |


## 3.npugraph\_ex特性功能介绍

npugraph\_ex提供多项功能，包括： [基础功能](#fig2)、[进阶功能](#fig3)和[问题定位](#fig4)。


**表 2**  npugraph_ex基础功能 <a id="fig2"></a>

<table align="left" border="1" cellpadding="6" cellspacing="0">
  <tr>
    <th align="left">功能</th>
    <th align="left">功能说明</th>
  </tr>
  <tr>
    <td align="left">force_eager功能</td>
    <td align="left">图执行前是否使用Eager模式运行。</td>
  </tr>
  <tr>
    <td align="left">force_recapture功能</td>
    <td align="left">是否强制每次执行时重新捕获NPUGraph。</td>
  </tr>
  <tr>
    <td align="left">FX图优化Pass配置功能</td>
    <td align="left">是否开启FX图优化能力。以减少计算过程中的内存搬运，从而提升性能。</td>
  </tr>
  <tr>
    <td align="left">FX图算子融合Pass配置功能</td>
    <td align="left">是否开启FX图算子融合Pass。该Pass基于已有Aten IR进行融合，从而提升性能。</td>
  </tr>
  <tr>
    <td align="left">NPUGraph间内存复用功能</td>
    <td align="left">NPUGraph间内存复用功能，支持多种模式。</td>
  </tr>
  <tr>
    <td align="left">静态Kernel编译功能</td>
    <td align="left">是否开启静态Kernel编译。</td>
  </tr>
  <tr>
    <td align="left">冗余算子消除功能</td>
    <td align="left">是否对冗余Kernel进行优化处理。</td>
  </tr>
  <tr>
    <td align="left">固定权重类输入地址功能</td>
    <td align="left">图执行时是否固定权重类输入地址。</td>
  </tr>
  <tr>
    <td align="left">重捕获次数限制功能</td>
    <td align="left">设置重捕获次数。</td>
  </tr>
  <tr>
    <td align="left">集合通信入图</td>
    <td align="left">实现集合通信算子Ascend Converter，调用torch.compile时默认已支持集合通信算子入图。</td>
  </tr>
  <tr>
    <td align="left">Cat算子消除功能</td>
    <td align="left">是否开启Cat算子消除优化以减少内存拷贝和临时张量分配，提升执行性能。</td>
  </tr>
  <tr>
    <td align="left">图捕获安全策略配置</td>
    <td align="left">控制图捕获过程中，对某些可能不安全操作（如分配device内存）的处理策略。</td>
  </tr>
</table>

**表 3**  npugraph_ex进阶功能 <a id="fig3"></a>

<table align="left" border="1" cellpadding="6" cellspacing="0">
  <tr>
    <th align="left">功能</th>
    <th align="left">功能说明</th>
  </tr>
  <tr>
    <td align="left">模型编译缓存功能</td>
    <td align="left">在推理服务和弹性扩容等业务场景中，使用编译缓存可有效缩短服务启动后的首次推理时延。</td>
  </tr>
  <tr>
    <td align="left">多流表达功能</td>
    <td align="left">大模型推理场景下，对于一些可并行的场景，可划分多个stream提升执行效率。</td>
  </tr>
  <tr>
    <td align="left">AI-Core和Vector-Core限核功能</td>
    <td align="left">提供Stream级核数配置，可调整最大AI Core数和Vector Core数，避免算子执行并行度降低。</td>
  </tr>
  <tr>
    <td align="left">自定义FX图优化Pass功能</td>
    <td align="left">传入自定义FX Pass函数，该配置可控制自定义Pass在框架内置Pass执行前/后生效。</td>
  </tr>
  <tr>
    <td align="left">SuperKernel功能</td>
    <td align="left">将连续的可融合Task合并为一个SuperKernel Task，减少任务调度开销，提升执行性能。</td>
  </tr>
</table>

**表 4**  npugraph\_ex 问题定位 <a id="fig4"></a>

<table align="left" border="1" cellpadding="6" cellspacing="0">
  <tr>
    <th align="left">功能</th>
    <th align="left">功能说明</th>
  </tr>
  <tr>
    <td align="left">图编译Debug信息保存功能</td>
    <td align="left">通过复用原生DEBUG环境变量TORCH\_COMPILE\_DEBUG开启日志打印和文件Dump。</td>
  </tr>
  <tr>
    <td align="left">算子Data-Dump功能</td>
    <td align="left">是否开启数据dump功能。</td>
  </tr>
  <tr>
    <td align="left">多流并发死锁检测功能</td>
    <td align="left">是否开启多流并发死锁检测功能。</td>
  </tr>
</table>

---
## 4.课后练习

(1)【单选题】关于 TorchAir 定位与两种图执行模式描述，错误的是？()
- A. TorchAir 是 torch_npu 配套的图模式扩展库，提供昇腾适配的 torch.compile 后端
- B. backend="npugraph_ex" 对应 aclgraph 捕获回放模式，一次捕获可多次执行，降低 Host 调度开销
- C. mode="max-autotune" 开启 GE 图模式，会把 PyTorch FX 图转为 Ascend IR，由 GE 图引擎编译执行
- D. npugraph_ex（aclgraph）执行模式依赖 GE 引擎完成图编译转换

(2)【多选题】下列属于 Eager 单算子执行模式（未使用 torch.compile）核心特征的有？()
- A. 即时执行，算子定义后立刻运行，无需提前构建完整计算图
- B. 动态计算图，每次运行分支、shape 变化都会重新生成计算图
- C. 延迟执行，先构建完整静态图再批量下发硬件执行
- D. 逐算子独立下发 NPU 调度，无全局图融合、图变换优化

 (3)【多选题】下列关于 npugraph_ex 基础功能与概念说法正确的是？()
- A. FX 算子融合 Pass、冗余算子消除、Cat 算子消除均属于 npugraph_ex 基础优化能力
- B. force_eager 功能可控制图执行前是否切换为 Eager 模式运行
- C. GE（Graph Engine）负责图优化、编译、执行控制，支持多框架图转 Ascend IR
- D. TorchAir 图模式是原生 PyTorch Eager 模式，不需要配合 torch.compile 使用


**运行以下代码单元查看参考答案与解析。**

In [ ]:
import os
answer_path = "answer/01.04_answer.txt"
if os.path.exists(answer_path):
    with open(answer_path, "r", encoding="utf-8") as f:
        print(f.read())
else:
    print("答案文件未找到，请检查 ./answer/ 目录结构。")